In [ ]:
# 스트림 방식으로 출력하기
# 일반적인 대화에서 스트림을 구현 해보았으나 랭체인의 도구(tool)을 이요할 때는 gpt의 응답이 단순히 텍스트형태가 아니라
# 어떤 함수에 어떤 인자를 넣어 실행할지를 판단해서 딕셔너리 형태로 넘어온다.
# 그래서 스트임 출력을 하려면 랭체인에서도 별도 처리를 해야 한다.
# 전에 코드를 수정하며 스트림 방식으로 변경 해보자

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='저는 잘 지내고 있습니다! 당신은 어떻게 지내고 계신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 12, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_255abcd69b', 'id': 'chatcmpl-DWYLmABnvpPvjP4itFJdSv6cKNKpZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da8a5-74df-79d0-a000-2d68b3c8a82c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 19, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_baaa53d70d', 'id': 'chatcmpl-DWYOuGQmxMYcXYtjjogFV6FwwSfFK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da8a8-6c73-7d63-95ac-7aa3600b9d30-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_O68esKaHYqeQDWWQWjnpAEl4', 'type': 'tool_call'}],

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-04-20 11:11:55 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_baaa53d70d', 'id': 'chatcmpl-DWYOuGQmxMYcXYtjjogFV6FwwSfFK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da8a8-6c73-7d63-95ac-7aa3600b9d30-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_O68esKaHYqeQDWWQWjnpAEl4', 'type': 'tool_call'}

In [6]:
llm_with_tools.invoke(messages)

AIMessage(content='부산의 현재 시각은 2026년 4월 20일 11시 11분 55초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 192, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DWYPFQc8YHDi865e222is09KuUUfi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da8a8-c0a6-7c32-997c-08203b1607fe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 29, 'total_tokens': 221, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [8]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 283, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe7a5277a8', 'id': 'chatcmpl-DWYPkKVNwnFeF0wrNoKAomtNYEBbe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019da8a9-3664-7a21-9e1e-fc2b45393d56-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_RcPGIH79hGQslRfOQNK82s0G', 'type': 'tool_call'}, {'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1d'}}, 'id': 'call_jeLxhLPzXFPPtQbOiC0FTwfr', 'type': 'tool_call'}, {'name': 'get_yf_stock

In [10]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-03-18 00:00:00-04:00 | 399    | 403.07 | 392.31 |  392.78 | 5.08531e+07 |           0 |              0 |\n| 2026-03-19 00:00:00-04:00 | 387.27 | 387.27 | 378.73 |  380.3  | 6.70783e+07 |           0 |              0 |\n| 2026-03-20 00:00:00-04:00 | 379.85 | 379.89 | 364.46 |  367.96 | 7.86286e+07 |           0 |              0 |\n| 2026-03-23 00:00:00-04:00 | 373.09 | 385.33 | 372.73 |  380.85 | 7.4606e+07  |           0 |              0 |\n| 2026-03-24 00:00:00-04:00 | 376.56 | 387.48 | 376.31 |  383.03 | 6.00049e+07 |           0 |              0 |\n| 2026-03-25 00:00:00-04:00 | 389.99 | 396.23 | 385.01 |  385.95 | 5.51573e+07 |           0 |              0 |\n| 2026-03-26 00:00:00-04:0

In [11]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전 테슬라(TSLA)의 주가는 다음과 같았습니다:\n\n- 2026년 3월 18일: 종가 392.78 달러\n\n현재 테슬라의 주가는 다음과 같습니다:\n\n- 2026년 4월 17일: 종가 400.62 달러\n\n따라서 테슬라는 한 달 전에 비해 주가가 **올랐습니다**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 1965, 'total_tokens': 2057, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe7a5277a8', 'id': 'chatcmpl-DWYQ4x4NKUbPlQB7ToowdrHMytWXs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da8a9-8959-7272-9293-72f124e5268d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1965, 'output_tokens': 92, 'total_tokens': 2057, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details

In [12]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 

|안|녕하세요|!| 한국| 사회|에는| 여러| 가지| 문제|점|이| 있습니다|.| 여기|서| 몇| 가지| 주요|한| 문제|를| 살|펴|볼|게|요|.

|1|.| **|청|년| 실|업|**|:| 청|년|층|의| 일|자|리가| 부족|하여| 많은| 젊|은| 사람들이| 취|업|에| 어려|움을| 겪|고| 있습니다|.| 이는| 고|용| 시장|의| 경|직|성과| 경제| 구조|의| 문제|로| 인해| 더욱| 심|화|되고| 있습니다|.

|2|.| **|주|거| 문제|**|:| 대|도|시|,| 특히| 서울|을| 중심|으로| 주|거|비|가| 급|격|히| 상승|하고| 있습니다|.| 이|로| 인해| 젊|은| 세|대|가| 주|택|을| 구|하기| 힘|들|어| 하고|,| 주|거| 불|안|정|성이| 커|지고| 있습니다|.

|3|.| **|노|인| 빈|곤|**|:| 한국|은| 고|령|화| 사회|로| 접|어|들|면서| 노|인| 빈|곤| 문제가| 심|각|하게| 대|두|되고| 있습니다|.| 노|후|에| 대한| 충분|한| 대비|가| 이루|어|지|지| 않은| 경우|가| 많|고|,| 이에| 따른| 사회|적| 지원| 체|계|의| 한|계|도| 드|러|나|고| 있습니다|.

|4|.| **|성|차|별|과| 성|폭|력|**|:| 성|희|롱|,| 성|폭|력| 사건|이| 빈|발|하고| 그|에| 대한| 사회|적| 인|식|이| 아직| 부족|한| 편|입니다|.| 이러한| 문제|는| 여성|의| 인|권|과| 안전|을| 위|협|하고| 있습니다|.

|5|.| **|정|신| 건강| 문제|**|:| 사회|의| 빠|른| 변화|와| 경쟁|적인| 환경| 속|에서| 정신| 건강| 문제가| 점|점| 더| 중요|해|지고| 있습니다|.| 이러한| 문제|에| 대한| 인|식| 부족|과| 치료| 접근|의| 어려|움|이| 있습니다|.

|6|.| **|학생| 과|중|한| 학|업| 부담|**|:| 교육| 경쟁|이| 치|열|한| 한국| 사회|에서는| 학생|들이| 지나|치|게| 많은| 학|업| 부담|을| 느|끼|고| 있으며|,| 이는| 스트|레스|와

In [ ]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)  
# tools와 연결된 llm_with_tools에서 함수를 호출하도록 질문
# 기존 코드에서는 .invoke()를 사용 했지만 지금은 .stream()을 사용함
# 단순한 텍스트 답변을 받을 때는 .stream()을 사용해도 문젱벗이 텍스트를 이어 붙여 최종 답변을 만든다.
# 하지만 지금처럼 도구가 연결된 상태에서는 어떤 함수를 어떤 값으로 실행할지에 관한 정보도 스트림 방식으로 반환됨
# 이 내용을 살펴보기 위해서 response를 for문으로 처리해서 순차적으로 넘어오는 결과를 출력 해보자.

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk  # 청크 누적용 코드
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)
# 랭체인으로 .stream()을 사용하면 결과가 AIMessageChunk 형식으로 반환됨
# AIMessageChunk는 개발자가 이전 조작을 쉽게 이어 붙일 수 있도록 도와주는 기능을 함
# for문에서 각 조각이 chunk변수에 담기고 + 연산자를 통해 이조각들이 gathered에 계속 추가 됨

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_c

In [ ]:
gathered # 출력 해서 확인 해보자.
# AIMessageChunk(content='', additional_kwargs={}, 
# response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 
# 'system_fingerprint': 'fp_fe7a5277a8', 'service_tier': 'default'}, id='lc_run--019da8ad-feb0-72c0-b0b8-a219352e88f8', 
# tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_call'}], 
# invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 226, 'input_token_details': 
# {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, 
# tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 
# 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe7a5277a8', 'service_tier': 'default'}, id='lc_run--019da8ad-feb0-72c0-b0b8-a219352e88f8', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 226, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [15]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-04-20 11:21:09 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe7a5277a8', 'service_tier': 'default'}, id='lc_run--019da8ad-feb0-72c0-b0b8-a219352e88f8', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 226, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_XsyZbOlf8tmPGQxnuY0ZurhF', 'index': 0, 'ty

In [16]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

|부|산|은| 현재| |202|6|년| |4|월| |20|일| |11|시| |21|분|입니다|.||||